# 05 RNN/LSTM ????

## ????

- ?????????????????
- ????????????????
- ?? LSTM ?????????
- ?? loss ????????????????

## ????

???????????????????????????????????????

LSTM ?????????????????????????????????????????????? `[batch, features]`??? `[batch, seq_len, features]`?

## ????

- ?????????????
- ?????????????????????????
- ?????????????????
- `batch_first=True`???????? `[batch, seq_len, features]`?

## ??????

???????????????????????? LSTM ???????????????

In [ ]:
import torch
from torch import nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

steps = np.linspace(0, 16 * np.pi, 420)
series = np.sin(steps) + 0.08 * np.random.randn(len(steps))
series = series.astype(np.float32)

plt.figure(figsize=(8, 3))
plt.plot(series)
plt.title("Synthetic sequence")
plt.xlabel("time")
plt.ylabel("value")
plt.grid(alpha=0.3)
plt.show()

## ????

????????? LSTM ????????????????????? shape ??????????????????????????

### ???????

??????????????????????? LSTM?????????? `x_train.shape`?`y_train.shape` ??? batch ??? shape?

In [ ]:
def make_windows(values, window_size=20):
    xs, ys = [], []
    for i in range(len(values) - window_size):
        xs.append(values[i:i + window_size])
        ys.append(values[i + window_size])
    x = torch.tensor(np.array(xs)).unsqueeze(-1)
    y = torch.tensor(np.array(ys)).unsqueeze(-1)
    return x, y

window_size = 20
x_all, y_all = make_windows(series, window_size)
split = int(len(x_all) * 0.8)
x_train, y_train = x_all[:split], y_all[:split]
x_test, y_test = x_all[split:], y_all[split:]

print("x_train shape:", x_train.shape)
print("y_train shape:", y_train.shape)

In [ ]:
class LSTMRegressor(nn.Module):
    def __init__(self, hidden_dim=32):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=hidden_dim, batch_first=True)
        self.output = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        sequence_output, _ = self.lstm(x)
        last_step = sequence_output[:, -1, :]
        return self.output(last_step)

model = LSTMRegressor(hidden_dim=32)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
losses = []

for epoch in range(80):
    model.train()
    pred = model(x_train)
    loss = criterion(pred, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    losses.append(loss.item())

model.eval()
with torch.no_grad():
    test_pred = model(x_test)
    test_loss = criterion(test_pred, y_test).item()

print(f"final train loss={losses[-1]:.4f}, test loss={test_loss:.4f}")

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(losses)
plt.title("LSTM training loss")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(8, 3))
plt.plot(y_test.numpy()[:120], label="target")
plt.plot(test_pred.numpy()[:120], label="prediction")
plt.title("One-step prediction")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## ????

??????????loss ????????????????????????????????????????????????????????????????????????????

## ????

- ???? feature ????? LSTM shape ????
- ???? `batch_first=True`???? `[batch, seq, feature]` ???
- ???????????????????????
- ??????loss ?????

????????????????????????????????????????????????????

In [ ]:
print("one sample input shape:", x_train[:1].shape)
print("one prediction shape:", model(x_train[:1]).shape)

## ????

**Q???????????**  
A???????????????????????????????????

**Q?LSTM ??? MLP ???**  
A?????LSTM ?????????????????????????????

**Q??????????????**  
A?????????????????????????????????????????

## ??

?????????????????????????????????????????????????